In [1]:
import pandas as pd
import numpy as np

# 1. 20 yıllık günlük tarih serisini oluştur (2004-01-01 -> 2023-12-31)
dates = pd.date_range(start="2004-01-01", end="2023-12-31", freq="D")
df = pd.DataFrame({'Tarih': dates})

# 2. Nüfus Artış Trendi (Yıllar içinde artan temel tüketim)
# 2004'te ~120.000 m3'ten 2023'te ~180.000 m3'e doğru doğrusal artış
base_consumption = np.linspace(120000, 180000, len(dates))

# 3. Mevsimsellik (Yaz aylarında artan, kışın azalan tüketim)
day_of_year = df['Tarih'].dt.dayofyear
# Sinüs dalgası: Yılın ~200. günlerinde (Temmuz/Ağustos) zirve yapar
seasonality = np.sin((day_of_year - 110) / 365.25 * 2 * np.pi) * 25000

# 4. Sıcaklık Verisi (Mevsimselliğe bağlı sentetik sıcaklık)
df['Ort_Sicaklik'] = 15 + np.sin((day_of_year - 110) / 365.25 * 2 * np.pi) * 16 + np.random.normal(0, 3, len(dates))
df['Ort_Sicaklik'] = df['Ort_Sicaklik'].round(1)

# Aşırı sıcak günlerde ekstra su tüketimi (Tarımsal sulama, soğutma vb.)
# Sıcaklık 30 derecenin üzerindeyse tüketime ekstra ekle
temp_spike_effect = np.where(df['Ort_Sicaklik'] > 30, (df['Ort_Sicaklik'] - 30) * 1500, 0)

# 5. Hafta Sonu Etkisi (Hafta sonları evsel su kullanımı genellikle artar)
df['Hafta_Sonu'] = df['Tarih'].dt.weekday.isin([5, 6]).astype(int)
weekend_effect = df['Hafta_Sonu'] * 4500

# 6. Rastgele Gürültü (Gerçek dünya dalgalanmaları)
noise = np.random.normal(0, 3000, len(dates))

# 7. Toplam Tüketimi Hesapla
df['Su_Tuketimi_m3'] = base_consumption + seasonality + temp_spike_effect + weekend_effect + noise
df['Su_Tuketimi_m3'] = df['Su_Tuketimi_m3'].round().astype(int)

# CSV Olarak Kaydet
file_name = "malatya_gunluk_su_tuketimi_20yil.csv"
df.to_csv(file_name, index=False)

print(f"{len(df)} satırlık veri seti '{file_name}' adıyla başarıyla oluşturuldu!")

7305 satırlık veri seti 'malatya_gunluk_su_tuketimi_20yil.csv' adıyla başarıyla oluşturuldu!


In [2]:
import pandas as pd
import numpy as np

# 1. 20 yıllık takvimi oluştur
dates = pd.date_range(start="2004-01-01", end="2023-12-31", freq="D")
df = pd.DataFrame({'Tarih': dates})
df['Ay'] = df['Tarih'].dt.month
df['Yil'] = df['Tarih'].dt.year

# 2. Malatya aylara göre yağış görme olasılığı
# Kış ve Bahar yüksek (Nisan %45), Yaz çok düşük (Temmuz %2)
rain_prob = {
    1: 0.35, 2: 0.35, 3: 0.40, 4: 0.45, 5: 0.40, 6: 0.15,
    7: 0.02, 8: 0.02, 9: 0.10, 10: 0.25, 11: 0.30, 12: 0.35
}

np.random.seed(42) # Sonuçların her çalışmada aynı çıkması için
base_rain = np.zeros(len(dates))

# 3. Günlük yağışları Üstel Dağılım (Exponential Distribution) ile oluştur
# Çoğu gün hafif yağış, nadiren şiddetli sağanak mantığına uyar.
for i, month in enumerate(df['Ay']):
    if np.random.rand() < rain_prob[month]:
        # scale=5.0 ortalama 5mm yağış üretir, ancak 20-30mm'lik ekstrem günler de çıkar
        base_rain[i] = np.random.exponential(scale=5.0)

# 4. Kuraklık Yılları Faktörü (Modelin ekstrem koşulları öğrenmesi için)
# Bu yıllarda düşen toplam yağışı %40 oranında baskılıyoruz
drought_years = [2007, 2008, 2014, 2020, 2021]
drought_mask = df['Yil'].isin(drought_years)
base_rain[drought_mask] *= 0.6 

# 5. Barajlara Göre Dağıtım (Yükselti ve havza farkı simülasyonu)
# Sürgü ve Sultansuyu dağlık alana daha yakın, yağış ortalamaları Karakaya'dan bir tık fazladır.
df['Karakaya_Yagis_mm'] = np.clip(base_rain * np.random.normal(0.9, 0.1, len(dates)), 0, None).round(1)
df['Surgu_Yagis_mm'] = np.clip(base_rain * np.random.normal(1.1, 0.15, len(dates)), 0, None).round(1)
df['Sultansuyu_Yagis_mm'] = np.clip(base_rain * np.random.normal(1.05, 0.1, len(dates)), 0, None).round(1)

# Genel Ortalama
df['Genel_Ortalama_mm'] = df[['Karakaya_Yagis_mm', 'Surgu_Yagis_mm', 'Sultansuyu_Yagis_mm']].mean(axis=1).round(1)

# Sadece gerekli kolonları tut
df = df.drop(columns=['Ay', 'Yil'])

# CSV olarak kaydet
file_name = "malatya_gunluk_yagis_20yil.csv"
df.to_csv(file_name, index=False)

print(f"{len(df)} satırlık veri seti '{file_name}' adıyla başarıyla oluşturuldu!")

7305 satırlık veri seti 'malatya_gunluk_yagis_20yil.csv' adıyla başarıyla oluşturuldu!


In [1]:
import pandas as pd
import numpy as np

# 1. Tarih serisi (Aylık bazda 30 yıl: 2024 - 2053)
dates = pd.date_range(start="2024-01-01", end="2053-12-31", freq="MS")
df = pd.DataFrame({'Tarih': dates})
df['Yil'] = df['Tarih'].dt.year
df['Ay'] = df['Tarih'].dt.month

# Zaman indeksi (0'dan 359'a kadar aylar)
t = np.arange(len(df))
yil_indeksi = t / 12  # 0'dan 30'a kadar küsuratlı yıl sayacı

# 2. Sıcaklık Projeksiyonu (30 yılda yaklaşık +2.0 derece artış)
# Temel mevsimsellik (Ocak: ~2C, Temmuz: ~29C)
base_temp = 15.5 - 13.5 * np.cos((df['Ay'] - 1) * 2 * np.pi / 12)
temp_trend = yil_indeksi * 0.066  
df['Ort_Sicaklik'] = np.round(base_temp + temp_trend + np.random.normal(0, 1.5, len(df)), 1)

# 3. Yağış Projeksiyonu (Yıllık %0.5 kümülatif azalış)
# Temel aylık yağış (mm) - Kış/Bahar yüksek, Yaz sıfıra yakın
base_precip = 45 + 25 * np.cos((df['Ay'] - 2) * 2 * np.pi / 12)
base_precip = np.clip(base_precip, 5, None) 
precip_decay = (1 - 0.005) ** yil_indeksi # Kümülatif azalma çarpanı
df['Toplam_Yagis'] = np.round((base_precip * precip_decay) + np.random.normal(0, 8, len(df)), 1)
df['Toplam_Yagis'] = np.clip(df['Toplam_Yagis'], 0, None) # Negatif yağış fiziksel olarak imkansız

# 4. Zirve Ayı (Kar Erimesi) Kayması
# İlk başta zirve Mayıs (Ay=5), 30 yıl sonra Mart sonu (Ay=3.5)
peak_month_shift = 5.0 - (yil_indeksi * (1.5 / 30))

# Fazı kaydırılmış kosinüs dalgası ile doluluk mevsimselliği
doluluk_mevsimsellik = -np.cos((df['Ay'] - peak_month_shift) * 2 * np.pi / 12)

# Karakaya Barajı (Genel Trend: Sıcaklık artışı buharlaşmayı artırır, yağış azalışı beslemeyi düşürür)
base_karakaya = 65.0
karakaya_trend = -(yil_indeksi * 0.4) # 30 yılda genelde -12 puanlık ortalama düşüş
df['Karakaya_Doluluk'] = np.round(base_karakaya + (doluluk_mevsimsellik * 15) + karakaya_trend + np.random.normal(0, 3, len(df)), 1)

# Sürgü Barajı (Tarımsal talebe daha duyarlı, yazın kuraklıktan daha sert etkilenecek)
base_surgu = 68.0
surgu_yaz_cezasi = np.where(df['Ort_Sicaklik'] > 28, (df['Ort_Sicaklik'] - 28) * 1.5, 0)
df['Surgu_Doluluk'] = np.round(base_surgu + (doluluk_mevsimsellik * 18) - surgu_yaz_cezasi - (yil_indeksi * 0.5) + np.random.normal(0, 3, len(df)), 1)

# Sultansuyu Barajı
base_sultansuyu = 64.0
sultansuyu_yaz_cezasi = np.where(df['Ort_Sicaklik'] > 28, (df['Ort_Sicaklik'] - 28) * 1.2, 0)
df['Sultansuyu_Doluluk'] = np.round(base_sultansuyu + (doluluk_mevsimsellik * 16) - sultansuyu_yaz_cezasi - (yil_indeksi * 0.45) + np.random.normal(0, 3, len(df)), 1)

# Maksimum %100 ve Minimum %0 sınırlandırması (Fiziksel kural)
for col in ['Karakaya_Doluluk', 'Surgu_Doluluk', 'Sultansuyu_Doluluk']:
    df[col] = np.clip(df[col], 0.0, 100.0)

df['Genel_Ortalama'] = np.round(df[['Karakaya_Doluluk', 'Surgu_Doluluk', 'Sultansuyu_Doluluk']].mean(axis=1), 1)

# CSV Kaydet
file_name = "malatya_iklim_senaryosu_2024_2053.csv"
df.to_csv(file_name, index=False)
print(f"30 yıllık gelecek projeksiyonu ({len(df)} ay) başarıyla '{file_name}' olarak kaydedildi.")

30 yıllık gelecek projeksiyonu (360 ay) başarıyla 'malatya_iklim_senaryosu_2024_2053.csv' olarak kaydedildi.


In [2]:
import pandas as pd
import numpy as np

# 1. Zaman serisi (2004-2023, Aylık)
dates = pd.date_range(start="2004-01-01", end="2023-12-31", freq="MS")
regions = ["Eski_Merkez", "Hizli_Buyuyen", "Yeni_Yerlesim"]

data = []
np.random.seed(42)

for current_date in dates:
    yil = current_date.year
    ay = current_date.month
    
    # Şubat 2023 Depremi Kontrolü
    deprem_sonrasi = 1 if current_date >= pd.to_datetime("2023-02-01") else 0
    
    for region in regions:
        # Bölgesel Başlangıç Değerleri (2004'ten itibaren)
        if region == "Eski_Merkez":
            # 2004'te nüfus durağan, son yıllarda hafif düşüş
            nufus = 280000 + ((yil - 2004) * 1500) if yil < 2018 else 300000 - ((yil - 2018) * 2000)
            altyapi_yasi = 18 + (yil - 2004) # 2004'te 18 yıllık
            sebeke_km = 800 + ((yil - 2004) * 2) # Yavaş büyüme
            basinc_bar = 3.8 + ((yil - 2004) * 0.02)
            deprem_hasar_carpani = 1.35 if deprem_sonrasi else 1.0 
            baz_kayip_kacak = 30.0 + ((yil - 2004) * 0.2) # Yaşlandıkça baz kayıp artıyor
            
        elif region == "Hizli_Buyuyen":
            # 2004'te seyrek, 2010 sonrası dikey patlama
            nufus = 80000 + ((yil - 2004) * 10000) 
            altyapi_yasi = max(0, yil - 2010) # Altyapının büyük kısmı 2010 sonrası yenilendi
            sebeke_km = 400 + ((yil - 2004) * 25) 
            # Dikey yapılar arttıkça basınç artıyor
            basinc_bar = 4.0 + ((yil - 2004) * 0.12)
            deprem_hasar_carpani = 1.20 if deprem_sonrasi else 1.0
            baz_kayip_kacak = 18.0 + ((yil - 2004) * 0.1)
            
        elif region == "Yeni_Yerlesim":
            # 2023 öncesi sadece köy/kasaba, deprem sonrası devasa TOKİ'ler
            nufus = 5000 + ((yil - 2004) * 200) if not deprem_sonrasi else 35000 + (current_date.month * 1000) # Deprem sonrası aydan aya artış
            altyapi_yasi = 1 if deprem_sonrasi else 10 + (yil - 2004)
            sebeke_km = 30 + ((yil - 2004) * 1) if not deprem_sonrasi else 120 + (current_date.month * 5)
            basinc_bar = 3.2 if not deprem_sonrasi else 3.8
            deprem_hasar_carpani = 1.0 # Yeni yapıldığı için deprem hasarı yok
            baz_kayip_kacak = 15.0 if not deprem_sonrasi else 8.0 # TOKİ'lerde baz kayıp düşük

        # Deprem anında (Şubat 2023) nüfus göçü simülasyonu
        if current_date >= pd.to_datetime("2023-02-01"):
            if region == "Eski_Merkez": nufus *= 0.75
            if region == "Hizli_Buyuyen": nufus *= 0.80
            
        # 1. Kayıp Kaçak Oranı Hesaplanması (Makine Öğrenmesi Hedef Değişkeni)
        # Formül: Baz + (Yaş Etkisi) + (Yüksek Basınç Cezası) * Deprem Çarpanı + Gürültü
        basinc_cezasi = max(0, basinc_bar - 4.5) * 3.0 # Basınç kırılganlığı artırıldı
        ya_etkisi = altyapi_yasi * 0.4 if region == "Eski_Merkez" else altyapi_yasi * 0.2
        
        kayip_orani = (baz_kayip_kacak + ya_etkisi + basinc_cezasi) * deprem_hasar_carpani
        
        # Yaz aylarında sulama vd. nedenlerle basınç düşer, sızıntı bir miktar azalır
        mevsimsel_basinc_farki = -2.5 if ay in [6,7,8] else 0 
        kayip_orani = np.clip(kayip_orani + mevsimsel_basinc_farki + np.random.normal(0, 2.0), 5, 75)
        
        # 2. Su Talebi ve Verilen Su Hesaplaması
        # Kişi başı ortalama günlük tüketim (Son yıllarda arttı)
        kisi_basi_tuketim = 160 + ((yil - 2004) * 2) 
        kisi_basi_tuketim_m3 = (kisi_basi_tuketim + 50) / 1000 if ay in [6,7,8,9] else kisi_basi_tuketim / 1000
        
        aylik_net_ihtiyac = nufus * kisi_basi_tuketim_m3 * 30
        
        # Verilen Su = Net İhtiyaç / (1 - Kayıp Oranı)
        aylik_verilen_su = aylik_net_ihtiyac / (1 - (kayip_orani / 100))
        
        data.append({
            "Tarih": current_date.strftime('%Y-%m-%d'),
            "Bolge_Tipi": region,
            "Nufus": int(nufus),
            "Altyapi_Yasi_Yil": round(altyapi_yasi, 1),
            "Sebeke_Uzunlugu_km": int(sebeke_km),
            "Ort_Basinc_Bar": round(basinc_bar, 1),
            "Deprem_Hasar_Katsayisi": round(deprem_hasar_carpani, 2),
            "Net_Tuketilen_Su_m3": int(aylik_net_ihtiyac),
            "Sisteme_Verilen_Su_m3": int(aylik_verilen_su),
            "Kayip_Kacak_Orani_%": round(kayip_orani, 1)
        })

df_altyapi = pd.DataFrame(data)
file_name = "malatya_kayip_kacak_20yil.csv"
df_altyapi.to_csv(file_name, index=False)
print(f"{len(df_altyapi)} satırlık 20 yıllık panel veri seti '{file_name}' başarıyla oluşturuldu.")

720 satırlık 20 yıllık panel veri seti 'malatya_kayip_kacak_20yil.csv' başarıyla oluşturuldu.


In [3]:
import pandas as pd
import numpy as np

# 1. 20 Yıllık Aylık Takvim
dates = pd.date_range(start="2004-01-01", end="2023-12-31", freq="MS")
data = []
np.random.seed(42)

for d in dates:
    y = d.year
    m = d.month
    
    # 2. Temel Kentsel Tüketim Büyümesi (Kışlık baz)
    urban_base = 3500000 + ((y - 2004) * 120000)
    
    # 3. Mevsimsel Sektörel Paylar ve Tarım Piki
    if m in [6, 7, 8, 9]: 
        # YAZ AYLARI: Tarımsal sulama patlaması (Kayısı bahçeleri vb.)
        total_volume = urban_base * 2.5 # Toplam su ihtiyacı 2.5 katına çıkar
        hane_ratio = 0.25
        sanayi_ratio = 0.08
        tarim_ratio = 0.60 # Aslan payı tarımın
        diger_ratio = 0.07
        
    elif m in [4, 5, 10]: 
        # BAHAR / GEÇİŞ AYLARI: Can suyu ve sonbahar sulaması
        total_volume = urban_base * 1.3
        hane_ratio = 0.50
        sanayi_ratio = 0.15
        tarim_ratio = 0.25
        diger_ratio = 0.10
        
    else: 
        # KIŞ AYLARI: Tarım minimumda, kentsel tüketim ağırlıklı
        total_volume = urban_base
        hane_ratio = 0.70
        sanayi_ratio = 0.18
        tarim_ratio = 0.02 # Sadece az miktarda sera / kapalı alan tarımı
        diger_ratio = 0.10
        
    # 4. Özel Şoklar (Anomaliler - 2020 Pandemi ve 2023 Depremi)
    if y == 2020 and m in [3, 4, 5]: 
        # Pandemi: Sanayi durdu, hane arttı, bahar tarımı devam etti
        hane_ratio = 0.60
        sanayi_ratio = 0.05
        tarim_ratio = 0.25
        diger_ratio = 0.10
        
    if y == 2023 and m >= 2: 
        # Deprem: Kentsel göç ve altyapı hasarı
        total_volume *= 0.65 
        if m in [6, 7, 8, 9]:
            # Yazın tarım devam etmek zorunda (ağaçlar kurumaması için) ama diğerleri düşük
            hane_ratio = 0.20
            sanayi_ratio = 0.05
            tarim_ratio = 0.65 
            diger_ratio = 0.10
        else:
            hane_ratio = 0.65
            sanayi_ratio = 0.05 # OSB hasarlı
            tarim_ratio = 0.05
            diger_ratio = 0.25 # Konteyner kentler, taşıma sular
        
    # 5. Gürültü (Noise) Eklenmesi ve Sektörel Dağıtım
    total = int(total_volume * np.random.normal(1.0, 0.03))
    
    hane = int(total * hane_ratio * np.random.normal(1.0, 0.01))
    sanayi = int(total * sanayi_ratio * np.random.normal(1.0, 0.01))
    tarim = int(total * tarim_ratio * np.random.normal(1.0, 0.02))
    
    # Kalan miktar "Diğer" kalemine atılır (Kusursuz denklik için)
    diger = total - (hane + sanayi + tarim)
    
    data.append([d.strftime("%Y-%m-%d"), total, hane, sanayi, tarim, diger])
    
# Veri Çerçevesi (DataFrame) Oluşturma
df = pd.DataFrame(data, columns=["Tarih", "Toplam_Tuketim_m3", "Hane_m3", "Sanayi_m3", "Tarim_m3", "Diger_m3"])

# CSV Olarak Kaydetme
file_name = "malatya_sektorel_tuketim_tarimli_20yil.csv"
df.to_csv(file_name, index=False)

print(f"{len(df)} aylık sektörel veri '{file_name}' başarıyla oluşturuldu!")

240 aylık sektörel veri 'malatya_sektorel_tuketim_tarimli_20yil.csv' başarıyla oluşturuldu!


In [4]:
import pandas as pd
import numpy as np

# 30 Yıllık Projeksiyon (2024 - 2053)
years = np.arange(2024, 2054)
data = []

# Başlangıç Değerleri (Deprem sonrası tahmini durum)
nufus = 750000 
hane_buyuklugu = 3.60
kentsel_oran = 0.82 # %82 şehir merkezi ve ilçeler, %18 köy/kırsal

for y in years:
    # 1. Nüfus Artış Senaryosu
    if y <= 2029:
        # Deprem sonrası geri dönüş ve TOKİ teslimatları (Hızlı artış)
        nufus *= 1.018 
    elif y <= 2038:
        # Normalleşme ve yavaşlama (Normal büyüme)
        nufus *= 1.008 
    else:
        # Demografik Plato (Nüfus artışı durma noktasına geliyor)
        nufus *= 1.002 

    # 2. Sosyolojik Değişimler
    # Hane halkı büyüklüğü her yıl doğrusal olarak azalarak Avrupa/Türkiye ortalamasına (2.8) yaklaşır
    hane_buyuklugu -= (3.60 - 2.80) / 30 
    
    # Kırsaldan kente göç devam eder, kentsel nüfus oranı %92'ye kadar çıkar
    kentsel_oran += (0.92 - 0.82) / 30

    # 3. Hesaplamalar
    kentsel_nufus = int(nufus * kentsel_oran)
    kirsal_nufus = int(nufus - kentsel_nufus)
    toplam_hane_sayisi = int(nufus / hane_buyuklugu) # Su idaresi için potansiyel abone sayısı

    data.append([
        y, 
        int(nufus), 
        kentsel_nufus, 
        kirsal_nufus, 
        round(hane_buyuklugu, 2), 
        toplam_hane_sayisi
    ])

# DataFrame Oluşturma ve Kaydetme
columns = ["Yil", "Toplam_Nufus", "Kentsel_Nufus", "Kirsal_Nufus", "Ortalama_Hane_Buyuklugu", "Toplam_Hane_Sayisi"]
df = pd.DataFrame(data, columns=columns)

file_name = "malatya_nufus_projeksiyonu_2024_2053.csv"
df.to_csv(file_name, index=False)

print(f"30 Yıllık Nüfus ve Hane Projeksiyonu '{file_name}' başarıyla oluşturuldu!")
print(f"2024 Hane Sayısı: {df.iloc[0]['Toplam_Hane_Sayisi']} -> 2053 Hane Sayısı: {df.iloc[-1]['Toplam_Hane_Sayisi']}")

30 Yıllık Nüfus ve Hane Projeksiyonu 'malatya_nufus_projeksiyonu_2024_2053.csv' başarıyla oluşturuldu!
2024 Hane Sayısı: 213666.0 -> 2053 Hane Sayısı: 330027.0


In [1]:
import pandas as pd
import numpy as np

# Zaman serisi (2004-2023, Yıllık)
years = np.arange(2004, 2024)
regions = ["Eski_Merkez", "Hizli_Buyuyen", "Yeni_Yerlesim"]

data = []
np.random.seed(42)

for yil in years:
    for region in regions:
        # Bölgesel Tarım Dinamikleri
        
        if region == "Eski_Merkez":
            # Battalgazi vb. Geleneksel kayısı bahçeleri nispeten korunuyor.
            # Şehirleşme daha yavaş, tarım alanı kaybı az.
            toplam_tarim = 250000 - ((yil - 2004) * 1500) # Her yıl 1500 dekar kayıp
            
            # Sulu tarıma geçiş (Baraj ve gölet yatırımları ile)
            sulu_tarim_orani = 0.40 + ((yil - 2004) * 0.015) 
            
            # Teknolojik dönüşüm: Vahşi sulamadan damla/yağmurlamaya geçiş
            damla_sulama = 2.0 + ((yil - 2004) * 2.8) 
            
        elif region == "Hizli_Buyuyen":
            # Yeşilyurt, Tecde vb. Çok ciddi tarım alanı kaybı (inşaat)
            toplam_tarim = 185000 - ((yil - 2004) * 3500) # Her yıl 3500 dekar betonlaşıyor
            
            # Kalan arazilerde daha yoğun sulu tarım yapılıyor
            sulu_tarim_orani = 0.45 + ((yil - 2004) * 0.02)
            
            # Şehir içi tarım olduğu için modern sulama daha yaygın
            damla_sulama = 5.0 + ((yil - 2004) * 3.0)
            
        elif region == "Yeni_Yerlesim":
            # Kırsal alanlar ve sonrasında TOKİ alanları
            if yil < 2023:
                # Deprem öncesi geniş mera ve kuru tarım arazileri
                toplam_tarim = 320000 - ((yil - 2004) * 500)
                sulu_tarim_orani = 0.15 + ((yil - 2004) * 0.01) # Genelde susuz tarım (buğday, arpa vb.)
                damla_sulama = 1.0 + ((yil - 2004) * 1.5)
            else:
                # Deprem sonrası (2023) TOKİ inşasıyla tarım alanlarının bir kısmı imara açıldı
                toplam_tarim = 320000 - ((2022 - 2004) * 500) - 15000 # Ani 15.000 dekar kayıp
                sulu_tarim_orani = 0.33 # Kalan araziler
                damla_sulama = 29.5

        # Hesaplamalar ve Mantıksal Sınırlar
        sulu_tarim_orani = min(sulu_tarim_orani, 0.85) # Sulu tarım %85'i geçemez
        damla_sulama = min(damla_sulama, 80.0) # Damla sulama %80'i geçemez
        
        sulu_tarim_dekar = int(toplam_tarim * sulu_tarim_orani)
        
        # Modele gürültü ekleyerek sentetik hissini azaltıyoruz
        toplam_tarim = int(toplam_tarim * np.random.normal(1.0, 0.01))
        sulu_tarim_dekar = int(sulu_tarim_dekar * np.random.normal(1.0, 0.015))
        
        data.append([
            yil, 
            region, 
            toplam_tarim, 
            sulu_tarim_dekar, 
            round(damla_sulama, 1)
        ])

# DataFrame Oluşturma
columns = ["Yil", "Bolge_Tipi", "Toplam_Tarim_Alani_Dekar", "Sulu_Tarim_Alani_Dekar", "Damla_Sulama_Orani_%"]
df = pd.DataFrame(data, columns=columns)

file_name = "malatya_bolgesel_tarim_arazileri_2004_2023.csv"
df.to_csv(file_name, index=False)

print(f"{len(df)} satırlık Bölgesel Tarım Arazi Verisi '{file_name}' başarıyla oluşturuldu!")

60 satırlık Bölgesel Tarım Arazi Verisi 'malatya_bolgesel_tarim_arazileri_2004_2023.csv' başarıyla oluşturuldu!
